<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

All dependencies present.


In [2]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [4]:
seed_val = 42
set_seed(seed_val)

In [12]:
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/Saved_variables/05-09-2024_all_images-20240509T194653Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated Images-20240523T233205Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated_Images_7-2-24-20240702T205017Z-001.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse.zip"

#!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Inverse_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular_resized.zip"

#!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Regular_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse_resized.zip"
#!echo "Unzip Complete"

In [5]:
# TODO: Replace curr_df_np126.pkl with images_unclassified

with open("./data/data_old/2024-08-18_curr_df_np126.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

feats_df.head()
#feats_df = feats_df.drop([700,1159,4061], axis = "rows")
#feats_df.reset_index(inplace = True, drop = True)
#feats_df["class"] = np.array(feats_df["class"].astype(int) -1).astype(int)
#print(np.unique(np.array(feats_df["class"]), return_counts = True))

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348


In [14]:
#with open("./data/data_old/2024-08-19_feat_array_unscaled.pkl", 'rb') as f:
#  feat_array_classified = pickle.load(f) # deserialize using load()

#with open("./data/data_old/2024-08-18_curr_df_np126.pkl", 'rb') as f:
#  df_classes_classified = pickle.load(f) # deserialize using load()

#feat_array_classified

In [6]:
# Features, 33k images
with open("./data/data_old/2024-08-19_feats_df_narrow_np126.pkl", 'rb') as f:
  feats_df_new = pickle.load(f) # deserialize using load()

feats_df_new

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,45.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,...,45.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0
3,0.036364,0.055723,0.088968,0.107719,-0.126763,0.011323,0.809702,1,2,11.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.985,0.0
4,0.035847,0.078762,0.07823,0.101485,-0.115752,0.010542,0.933894,1,2,12.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.969,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33765,0.047659,0.179187,0.122823,0.048745,-0.080442,0.011393,0.468993,0,3,noise 3 21522.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
33766,0.049658,0.087822,0.124444,0.105756,-0.140723,0.008773,1.001359,1,7,noise 7 21167.0.png,...,135.288,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.920,0.0
33767,0.025269,0.096806,0.0973,0.124239,-0.046031,0.008247,0.752525,0,5,noise 5 21035.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
33768,0.029474,0.085933,0.103994,0.103636,-0.119675,0.008054,1.058641,1,3,noise 3 24575.0.png,...,135.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.795,0.0


In [7]:
feats_df_new

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,45.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,...,45.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0
3,0.036364,0.055723,0.088968,0.107719,-0.126763,0.011323,0.809702,1,2,11.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.985,0.0
4,0.035847,0.078762,0.07823,0.101485,-0.115752,0.010542,0.933894,1,2,12.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.969,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33765,0.047659,0.179187,0.122823,0.048745,-0.080442,0.011393,0.468993,0,3,noise 3 21522.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
33766,0.049658,0.087822,0.124444,0.105756,-0.140723,0.008773,1.001359,1,7,noise 7 21167.0.png,...,135.288,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.920,0.0
33767,0.025269,0.096806,0.0973,0.124239,-0.046031,0.008247,0.752525,0,5,noise 5 21035.0.png,...,135.000,0.0,200.0,0.0,1.000,0.0,1.000,0.0,1.000,0.0
33768,0.029474,0.085933,0.103994,0.103636,-0.119675,0.008054,1.058641,1,3,noise 3 24575.0.png,...,135.000,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.795,0.0


In [8]:
feats_df.head()

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348


In [9]:
# Sets pred-class column equal to class column
feats_df["pred_class"] = feats_df["class"]

In [10]:
# two dataframes 
# Possibly df1 = feats_df_new
df1 = feats_df[["noise","path","seed","dir","pred_class"]]
df2 = feats_df[["noise","path","seed","dir","pred_class"]]

# concat's predicted and manual classified dataframes (df_classes_classified + feats_df)
pred_and_manual_df = pd.concat([df1, df2], axis=0)

pred_and_manual_df.reset_index(inplace = True, drop = True)
feats_df_new.reset_index(inplace = True, drop = True)

pred_and_manual_df.tail(20)

,noise,path,seed,dir,pred_class
64400,4,noise 4 22201.0.png,22201.0,/content/Images1/,None
64401,3,noise 3 24788.0.png,24788.0,/content/Images1/,1
64402,5,noise 5 24550.0.png,24550.0,/content/Images1/,1
64403,3,noise 3 23212.0.png,23212.0,/content/Images1/,None
64404,4,noise 4 20924.0.png,20924.0,/content/Images1/,None
64405,6,noise 6 20401.0.png,20401.0,/content/Images1/,1
64406,3,noise 3 22691.0.png,22691.0,/content/Images1/,1
64407,6,noise 6 24543.0.png,24543.0,/content/Images1/,None
64408,6,noise 6 22123.0.png,22123.0,/content/Images1/,1
64409,5,noise 5 24768.0.png,24768.0,/content/Images1/,1


In [11]:
#add full path to each dataframe to allow matching based on unique path
# df with all params: feats_df_new
# df with predicted classes: pred_and_manual_df

fdf_full_path = []
for i in range(feats_df_new.shape[0]):
  fdf_full_path.append(feats_df_new["dir"][i] + feats_df_new["path"][i])


cdf_full_path = []
for j in range(pred_and_manual_df.shape[0]):
  cdf_full_path.append((pred_and_manual_df["dir"][j] + pred_and_manual_df["path"][j]))


feats_df_new["full_path"] = fdf_full_path
pred_and_manual_df["full_path"] = cdf_full_path

In [12]:
feats_df.head(20)

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,2,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,None,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,2,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,None,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,2,-31.683971,-14.198348
5,None,2,5.png,5,/content/Images3/,None,28.932574,-14.527230
6,None,2,6.png,6,/content/Images3/,None,1.895376,-5.157169
7,None,2,7.png,7,/content/Images3/,None,32.919621,-18.103002
8,None,2,8.png,8,/content/Images3/,None,31.071663,-16.434768
9,None,2,10.png,10,/content/Images3/,None,28.874940,-14.434144


In [13]:
## Indices 1 - ALl elements in feats_df_new that match full paths with rows in pred_and_manual_df.


indices_1 = []
indices_2 = []
errors = 0
counter = 0
for i in range(feats_df_new.shape[0]):
  #pull up path to search for
  path = pred_and_manual_df["full_path"][i]

  #find index within feats_df that matches path
  ind_df = feats_df_new[feats_df_new["full_path"] == path]
  #print(ind_df.head())
  #print("Ind_df.shape[0] ", ind_df.shape[0])
    
  if ind_df.shape[0] == 1:
    ind = ind_df.index[0]
    # print(ind_df.head())
#    #print("ind: " + str(ind) + " i: " + str(i))
    indices_1.append(ind)
    indices_2.append(i)
  else:
   # indices.append("error")
   # print("error")
    errors += 1


  if counter % 1000 == 0:
    print(counter)
  counter += 1

#print(ind_df.head())
#print(indices_2)
print("Complete.")

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
Complete.


In [14]:
# match dfs
# drops all values of feats_df_new that aren't in indices_1
# drops all values of pred_and_manual_df that aren't in indices_2

feats_df_new = feats_df_new.iloc[indices_1]
pred_and_manual_df = pred_and_manual_df.iloc[indices_2]
feats_df_new.reset_index(inplace = True, drop = True)
pred_and_manual_df.reset_index(inplace = True, drop = True)

In [15]:
same = list(np.array(feats_df_new["full_path"]) == np.array(pred_and_manual_df["full_path"]))

print(same.count(True))
print(same.count(False))
print(len(same))

#print(same)

28659
0
28659


In [16]:
feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]

C:\Users\user.SR7872\AppData\Local\Temp\ipykernel_9236\3775421508.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]


In [17]:
#its a feats_df of constrained parameters
feats_df_new.head()

feats_df = feats_df_new

In [18]:
print(feats_df.shape)

(28659, 113)


In [19]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

XGBOOST

In [20]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
#! pip install graphviz
#! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


All dependencies present


In [21]:
np.unique(feats_df["num_spots"].astype(float))

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
       143., 144., 145., 146., 147., 148., 149., 15

In [22]:
# scale data 
# uses Standard Scaler, range [-1, 1]

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling = StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [23]:
print(X_test)
#print(y_train)

[[-0.64846417 -0.95141314 -0.78175594 ... -0.27656263  0.56365538
  -0.24122377]
 [ 0.30827125  0.39681846  1.27917161 ... -0.27656263 -4.37520359
  -0.24122377]
 [-0.63277998  1.33653853  1.27917161 ... -0.27656263  0.56365538
  -0.24122377]
 ...
 [-0.64846417 -0.95141314 -0.78175594 ... -0.27656263  0.56365538
  -0.24122377]
 [ 2.58247841 -0.5522239  -0.78175594 ... -0.27656263 -0.8978437
  -0.24122377]
 [ 0.872902   -0.82186124 -0.78175594 ... -0.27656263  0.08488844
  -0.24122377]]


In [24]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
20241,0.036670,0.073226,0.040154,0.061430,0.009830,0.850876,-0.101614
14778,0.010814,0.077545,0.084799,0.106047,0.011972,0.344742,-0.120386
25231,0.031524,0.146835,0.162371,0.089494,0.009966,0.948477,-0.151192
26948,0.032629,0.072921,0.088104,0.153645,0.009018,0.894565,-0.164564
12272,0.011581,0.058755,0.142204,0.123917,0.011563,1.160790,-0.115591
...,...,...,...,...,...,...,...
13827,0.008961,0.052993,0.128148,0.147533,0.010953,0.754449,-0.150162
2855,0.019784,0.072723,0.137530,0.070223,0.011001,0.669717,-0.102735
16710,0.019573,0.059808,0.089377,0.160549,0.010973,0.480674,-0.144362
25107,0.038058,0.123075,0.108594,0.162017,0.008668,0.761952,-0.113683


In [37]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:18:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:49: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:18:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:18:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "num_boost_round" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None, ...)

In [25]:
loaded_model = xgb.XGBRegressor()

loaded_model.load_model('./content/Models/XGB_model.model')

#loaded_model.fit(X_train, y_train)

# make predictions
y_pred = loaded_model.predict(X_test)

pd.DataFrame(data=y_pred) 

# Columns - parameters 1-7
# Rows - Subsample of rows

,0,1,2,3,4,5,6
0,0.033395,0.087170,0.064362,0.110597,0.010077,0.807379,-0.129942
1,0.011164,0.086442,0.089399,0.117670,0.010302,0.697959,-0.130809
2,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
3,0.035866,0.078264,0.087985,0.120647,0.009795,0.673017,-0.159450
4,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
...,...,...,...,...,...,...,...
5727,0.011736,0.069977,0.117183,0.112757,0.009499,1.038847,-0.129972
5728,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
5729,0.033395,0.087170,0.064362,0.110597,0.010077,0.807379,-0.129942
5730,0.038365,0.133653,0.101905,0.119501,0.009474,0.620962,-0.141445


In [41]:
## Save XGBoost Model to File

path = "./content/Models/XGB_model.model"

model.save_model(path)

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [14:36:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


In [27]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(5732, 7)
(5732, 7)
[32.13409601 54.19867644 25.56774297 40.13796377  8.68439102 37.65799152
 33.26148726]


In [28]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("./content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  if path != ".ipynb_checkpoints":
    real_imj_df["path"].append(path)
    real_imj_df["feats_df"].append(pd.read_csv("./content/ImageJ_data/Regular_resized/" + path))
    real_imj_df["inverse_feats_df"].append(pd.read_csv("./content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [45]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None) 
real_df.shape

columns0 = ['feats_df', 'inverse_feats_df']

real_df2 = real_df.drop(columns=columns0)

real_df2

#real_df.head(14)

,path,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,pred_params
0,bHLH2_OE_(green)_(0-141_thresh.tif.csv,30,67.001,0,350.333333,436.638320,104.649567,63.380542,69.287167,49.347736,105.791967,105.323220,95.300000,64.799254,54.300000,45.721002,18.366667,14.716166,31.333333,28.248107,28.469100,22.259388,11.132167,7.572338,95.829533,41.340200,0.519500,0.271161,34.549767,29.326032,8.933500e+04,1.113428e+05,100.0,0.0,8.933500e+04,1.113428e+05,97.966667,65.061757,61.133333,49.947795,106.094367,30.422104,14.895000,11.599036,2.534433,1.161018,0.472233,0.188002,0.736767,0.158909,3,187.999,255,9830.000000,13840.207585,95.572667,64.063517,36.149333,47.039893,479.711333,643.458785,59.666667,76.054074,0.000000,0.000000,72.333333,90.282273,71.000000,91.224997,70.875333,88.047917,67.994000,87.418781,165.417333,15.370366,0.643667,0.321746,100.305000,128.089096,2.506650e+06,3.529253e+06,100.0,0.0,2.506650e+06,3.529253e+06,59.666667,76.054074,67.000000,94.046088,118.021000,52.215878,71.000000,91.224997,1.264667,0.176532,0.808000,0.123701,0.839333,0.073436,"[0.02614826, 0.062585935, 0.12087097, 0.08383078, 0.010859064, 0.85632795, -0.09742121]"
1,bHLH2_RNAi_(green)_(0-170).tif.csv,0,0.000,0,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,800.000000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.785000,0.000000,282.843000,0.000000,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1,255.000,255,40000.000000,0.000000,100.000000,0.000000,100.000000,0.000000,797.657000,0.000000,0.000000,0.000000,0.000000,0.000000,200.000000,0.000000,200.000000,0.000000,225.676000,0.000000,225.676000,0.000000,0.000000,0.000000,0.790000,0.000000,282.843000,0.000000,1.020000e+07,0.000000e+00,100.0,0.0,1.020000e+07,0.000000e+00,0.000000,0.000000,0.000000,0.000000,135.000000,0.000000,200.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,"[0.023517208, 0.057141475, 0.07845976, 0.1292734, 0.010278776, 0.76815784, -0.08877929]"
2,F2_260_green)_(0-105).tif.csv,26,70.966,0,428.153846,519.658223,103.716385,53.872307,86.366654,56.292048,99.326923,90.300749,95.576923,54.687344,69.500000,53.847541,17.000000,14.665501,35.038462,27.607804,32.030154,22.516273,11.821077,7.936903,83.427346,31.445279,0.545500,0.229855,37.386077,29.418346,1.091792e+05,1.325128e+05,100.0,0.0,1.091792e+05,1.325128e+05,98.538462,54.670503,87.961538,54.2632

In [30]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

0
Counting done


In [31]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [32]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [33]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

0
Counter complete


In [34]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [35]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [36]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean', 'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean', 'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean', 'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std', 'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean', 'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std', 'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std', 'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std', 'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean', 'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted', 'Mean_inverted', 'Median_inverted', 'Area_inverted_mean', 'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std', 'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean', 'Perim._inverted_std', 'BX_inverted_mean', 'BX_inverted_std', 'BY_inverted_mean', 'BY_inverted_std', 'Width_inverted_mean',
       'Width_

In [ ]:
real_df.head()

In [38]:
# select features of real images
real_feats = real_df[real_df.columns[3:101]]

#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(real_feats)
real_feats_scaled = scaling.transform(real_feats)



# predict params of images from real feats
y_pred = loaded_model.predict(real_feats_scaled)

real_df["pred_params"] = list(y_pred)

In [39]:
real_df["pred_params"][0]

array([ 0.02614826,  0.06258593,  0.12087097,  0.08383078,  0.01085906,
        0.85632795, -0.09742121], dtype=float32)

In [40]:
real_df["pred_params"][0]

array([ 0.02614826,  0.06258593,  0.12087097,  0.08383078,  0.01085906,
        0.85632795, -0.09742121], dtype=float32)

In [41]:
## TODO: Grab dataframe at the top (w/ all the features), drop all unnessesary columns (diff. from real_df, drop parameters)
## choose first five images and run it and see how close they are. 

for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])

bHLH2_OE_(green)_(0-141_thresh.tif.csv
[ 0.02614826  0.06258593  0.12087097  0.08383078  0.01085906  0.85632795
 -0.09742121]
bHLH2_RNAi_(green)_(0-170).tif.csv
[ 0.02351721  0.05714148  0.07845976  0.1292734   0.01027878  0.76815784
 -0.08877929]
F2_260_green)_(0-105).tif.csv
[ 0.02331874  0.06008431  0.12086973  0.11648673  0.00954962  0.70141786
 -0.0975161 ]
F2_A 8_(green)_(0-100).tif.csv
[ 0.02924089  0.06581408  0.1192591   0.10536818  0.00979359  0.6629838
 -0.06067621]
LF10_11-20-23_(green)_(0-152).tif.csv
[ 0.0221226   0.10830527  0.12630259  0.11860634  0.0105213   0.6667085
 -0.10096822]
MLC_F1_11-20-23_(green)_(0-105).tif.csv
[ 0.02128733  0.12389465  0.09693768  0.12160463  0.01039962  0.81738204
 -0.09693675]
mpar_rto(c-c)_(green)_(0-137).tif.csv
[ 0.02338564  0.11086314  0.10433803  0.11101844  0.01052789  0.75643164
 -0.08920884]
mpar_rto_crispr_(green)_(0-155).tif.csv
[ 0.01959674  0.11570201  0.12073527  0.10527825  0.00973131  0.9733093
 -0.08260424]
mpar_wt_(green)_